<img src="../../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../../images/Konstanz_Logo.svg" width="200" /> <img src="../../images/KIT_Logo.png" width="200" />

# Cross Validation and Grid Search

Two problems are left over from [notebook
02](02_Linear_Regression.ipynb).

The first is that our verdict rested on **a single split** of the data. 20% of the
rides were held back, and the model was judged on those. Had a different 20% been
drawn, the score would have come out differently — perhaps flatteringly so.

The second is that most algorithms have **settings** that are not learned from the
data but chosen by hand, and different settings can change the result completely.

This notebook covers the standard answer to both: **cross validation** for a score
you can trust, and **grid search** for choosing the settings.

---

## Contents

1. [The data](#data)
2. [Why one split is not enough](#why)
3. [k-fold cross validation](#kfold)
4. [Grid search](#grid)
5. [The best model](#best)

<a id="data"></a>
## 1. The data

The same taxi rides and the same three features as in notebook 02, so that the
results are directly comparable. This notebook is self-contained and can be run on
its own.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet('../../files/green_tripdata_2023-01.parquet', engine='pyarrow')
df = df.sample(1000, random_state=42)

df['pickup_hour'] = df['lpep_pickup_datetime'].dt.hour
df['ride_duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
df['ride_duration_minutes'] = df['ride_duration'].dt.total_seconds().div(60).astype(int)

X = df[['trip_distance', 'pickup_hour', 'ride_duration_minutes']]
y = df['tip_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("training rows:", len(X_train))
print("test rows:    ", len(X_test))

training rows: 800
test rows:     200


<a id="why"></a>
## 2. Why one split is not enough

Holding back the last 20% and scoring the model on it gives **one** number, produced
by **one** arbitrary division of the data. That division might happen to be easy, or
happen to be hard. Either way we would not know.

It also creates a subtler danger. As soon as you start adjusting a model to improve
that score, the test data is influencing your decisions — and it stops being data
the model has never seen.

The usual remedy is a three-way division: **training** data to learn from,
**validation** data to compare candidates against, and **test** data that is touched
exactly once, at the very end, after everything has been decided.

<a id="kfold"></a>
## 3. k-fold cross validation

Cross validation avoids setting aside a permanent validation set. Instead the
training data is cut into **k** equal parts, and the whole procedure is run k times:
each time a different part is used for evaluation and the remaining parts for
training.

![k-fold cross validation](../../images/ml_kfold.svg)

Every row therefore serves as test data exactly once, and the k scores are averaged
into a single figure. That average is far more stable than any individual split,
because a lucky or unlucky division averages out.

`cross_val_score` does all k rounds in one call.

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

scores = cross_val_score(LinearRegression(), X_train, y_train,
                         scoring='neg_mean_squared_error', cv=5)

scores.round(3)

array([-4.603, -4.481, -4.943, -5.242, -6.886])

Note the name of the scoring rule: **`neg_mean_squared_error`**, not
`mean_squared_error`. scikit-learn's convention is that a score is always something
to be *maximised*, so an error has its sign flipped — the value closest to zero is
the best one. Passing `'mean_squared_error'` here is simply rejected.

Turning those five numbers back into an RMSE gives something comparable with
notebook 02.

In [3]:
print("RMSE per fold:", np.sqrt(-scores).round(3))
print(f"average RMSE: {np.sqrt(-scores).mean():.3f}")

RMSE per fold: [2.145 2.117 2.223 2.29  2.624]
average RMSE: 2.280


The spread between the folds is the point. It shows how much the score would have
moved had we happened to pick a different single split — which is exactly the
uncertainty a single number hides.

<a id="grid"></a>
## 4. Grid search

Now the second problem: settings that have to be chosen rather than learned.

To have something worth tuning we switch models. **Support vector regression**
(`SVR`) can fit curved relationships, and it has a parameter `C` controlling how
hard it tries to fit every training point — small values keep it simple, large
values let it follow the data closely.

There is no formula for a good `C`. You try several and see.

`GridSearchCV` automates exactly that: give it a list of values, and it runs a full
cross validation for each one.

In [4]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

grid_model = GridSearchCV(SVR(), param_grid=param_grid,
                          scoring='neg_mean_squared_error', cv=5)
grid_model.fit(X_train, y_train)

grid_model.best_estimator_

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


Seven values, five folds each — **35 models** were trained to produce that one
answer. This is why grid search grows expensive quickly: a second parameter with
seven values of its own would mean 245 fits, and the cost multiplies with every
parameter added.

The score of every candidate is kept, so the whole comparison can be inspected.

In [5]:
results = pd.DataFrame({
    "C": param_grid['C'],
    "RMSE": np.sqrt(-grid_model.cv_results_['mean_test_score']).round(3),
})

results

,C,RMSE
0,0.001,2.381
1,0.010,2.378
2,0.100,2.357
3,1.000,2.316
4,10.000,2.278
5,100.000,2.304
6,1000.000,2.386


<a id="best"></a>
## 5. The best model

`GridSearchCV` refits the winning configuration on the whole training set, so
`grid_model` can be used to predict directly.

This is the moment the test data is finally used — once, after every decision has
been made.

In [6]:
from sklearn.metrics import mean_squared_error

prediction = grid_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, prediction))

print(f"average tip:        {y.mean():.3f}")
print(f"RMSE, tuned SVR:    {rmse:.3f}")

average tip:        2.025
RMSE, tuned SVR:    2.254
